# Data Validation — Great Expectations

**Project:** Retail Sales ETL Pipeline  

This notebook validates the cleaned retail transactions dataset (`retail_transactions_clean.csv`) using Great Expectations before it is indexed into Elasticsearch. The goal is to ensure data integrity, correct types, and no corrupted values that could break downstream analytics or Kibana dashboards.

## 1. Import Libraries

In [1]:
import warnings
warnings.filterwarnings('ignore')

import great_expectations as ge
from great_expectations.data_context import FileDataContext
import pandas as pd

print("Great Expectations version:", ge.__version__)

Great Expectations version: 0.18.19


## 2. Instantiate Data Context

In [2]:
context = FileDataContext.create(project_root_dir='.')

print("GX Context created successfully!")
print(f"Context type: {type(context)}")

GX Context created successfully!
Context type: <class 'great_expectations.data_context.data_context.file_data_context.FileDataContext'>


## 3. Connect to Datasource

In [3]:
# Define datasource
datasource_name = 'retail-clean-data'
datasource = context.sources.add_or_update_pandas(datasource_name)

# Define data asset
asset_name = 'retail-transactions-clean'
path_to_data = '../data/processed/retail_transactions_clean.csv'
asset = datasource.add_csv_asset(asset_name, filepath_or_buffer=path_to_data)

# Build batch request
batch_request = asset.build_batch_request()

print("Datasource connected successfully!")
print(f"Datasource name : {datasource_name}")
print(f"Asset name      : {asset_name}")
print(f"File            : {path_to_data}")

Datasource connected successfully!
Datasource name : retail-clean-data
Asset name      : retail-transactions-clean
File            : ../data/processed/retail_transactions_clean.csv


## 4. Create Expectation Suite

In [4]:
suite_name = 'retail_validation_suite'
context.add_or_update_expectation_suite(suite_name)

validator = context.get_validator(
    batch_request=batch_request,
    expectation_suite_name=suite_name
)

print("Expectation Suite created!")
print(f"Suite name : {suite_name}")
print(f"\nData preview:")
print(validator.head())

Expectation Suite created!
Suite name : retail_validation_suite

Data preview:


Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

  invoice_no customer_id  gender  age payment_method  category  quantity  \
0    I138884     C241288  Female   28    Credit Card  Clothing         5   
1    I317333     C111565    Male   21     Debit Card     Shoes         3   
2    I127801     C266599    Male   20           Cash  Clothing         1   
3    I173702     C988172  Female   66    Credit Card     Shoes         5   
4    I337046     C189076  Female   53           Cash     Books         4   

     price invoice_date   shopping_mall  
0  1500.40   2022-08-05          Kanyon  
1  1800.51   2021-12-12  Forum Istanbul  
2   300.08   2021-11-09       Metrocity  
3  3000.85   2021-05-16    Metropol AVM  
4    60.60   2021-10-24          Kanyon  


## 5. Define Expectations

### Expectation 1: Column Values To Be Unique (`invoice_no`)

Every `invoice_no` must be unique since it is the primary key of the dataset. Duplicate invoice numbers would indicate data integrity issues or duplicate records, which could distort transaction counts and revenue calculations.

In [5]:
result_1 = validator.expect_column_values_to_be_unique('invoice_no')
print(result_1)

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  },
  "success": true,
  "meta": {},
  "result": {
    "element_count": 99457,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "expectation_config": {
    "meta": {},
    "expectation_type": "expect_column_values_to_be_unique",
    "kwargs": {
      "column": "invoice_no",
      "batch_id": "retail-clean-data-retail-transactions-clean"
    }
  }
}


### Expectation 2: Column Values To Be Between (`age`)

Every value in `age` must be between `18` and `70`. This ensures the dataset targets adult shoppers, ages below 18 indicate minors, and ages above 70 are statistically rare in shopping mall data. Values outside this range likely indicate data entry errors.

In [6]:
result_2 = validator.expect_column_values_to_be_between(
    'age',
    min_value=18,
    max_value=70
)
print(result_2)

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  },
  "success": true,
  "meta": {},
  "result": {
    "element_count": 99457,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "expectation_config": {
    "meta": {},
    "expectation_type": "expect_column_values_to_be_between",
    "kwargs": {
      "min_value": 18,
      "max_value": 70,
      "column": "age",
      "batch_id": "retail-clean-data-retail-transactions-clean"
    }
  }
}


### Expectation 3:  Column Values To Be In Set (`payment_method`)

Every value in `payment_method` must be one of `Cash`, `Credit Card`, or `Debit Card`, the only payment methods supported by the mall system. Any value outside this set indicates invalid or corrupted data that could distort payment analytics.

In [7]:
result_3 = validator.expect_column_values_to_be_in_set(
    'payment_method',
    ['Cash', 'Credit Card', 'Debit Card']
)
print(result_3)

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  },
  "success": true,
  "meta": {},
  "result": {
    "element_count": 99457,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "expectation_config": {
    "meta": {},
    "expectation_type": "expect_column_values_to_be_in_set",
    "kwargs": {
      "column": "payment_method",
      "value_set": [
        "Cash",
        "Credit Card",
        "Debit Card"
      ],
      "batch_id": "retail-clean-data-retail-transactions-clean"
    }
  }
}


### Expectation 4: Column Values To Be In Type List (`age`)

Every value in `age` must be of integer type (`INT`, `int64`, or `int32`). Age should always be a whole number, and if stored as float or string, it indicates that the data cleaning pipeline did not correctly cast the type.

In [8]:
result_4 = validator.expect_column_values_to_be_in_type_list(
    'age',
    ['INT', 'INTEGER', 'int', 'int64', 'int32']
)
print(result_4)

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

{
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  },
  "success": true,
  "meta": {},
  "result": {
    "observed_value": "int64"
  },
  "expectation_config": {
    "meta": {},
    "expectation_type": "expect_column_values_to_be_in_type_list",
    "kwargs": {
      "column": "age",
      "type_list": [
        "INT",
        "INTEGER",
        "int",
        "int64",
        "int32"
      ],
      "batch_id": "retail-clean-data-retail-transactions-clean"
    }
  }
}


### Expectation 5: Column Standard Deviation To Be Between (`price`)

The standard deviation of `price` must be between `1` and `942`. A stdev of 0 would mean all prices are identical (impossible in a real retail dataset), while an extremely high stdev indicates extreme outliers or data corruption. The upper bound of 942 is derived from the historical dataset stdev of 941.18.

In [9]:
result_5 = validator.expect_column_stdev_to_be_between(
    'price',
    min_value=1,
    max_value=942
)
print(result_5)

Calculating Metrics:   0%|          | 0/4 [00:00<?, ?it/s]

{
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  },
  "success": true,
  "meta": {},
  "result": {
    "observed_value": 941.184567215467
  },
  "expectation_config": {
    "meta": {},
    "expectation_type": "expect_column_stdev_to_be_between",
    "kwargs": {
      "min_value": 1,
      "max_value": 942,
      "column": "price",
      "batch_id": "retail-clean-data-retail-transactions-clean"
    }
  }
}


### Expectation 6: Column Values To Match Regex (`invoice_no`)

All values in `invoice_no` must match the format `I` followed by digits (e.g., `I138884`). Invoice numbers in this retail system follow a consistent format, in which any value that does not match indicates a data entry error, a corrupted record, or data from an incompatible source.

In [10]:
result_6 = validator.expect_column_values_to_match_regex(
    'invoice_no',
    r'^I\d+$'
)
print(result_6)

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  },
  "success": true,
  "meta": {},
  "result": {
    "element_count": 99457,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "expectation_config": {
    "meta": {},
    "expectation_type": "expect_column_values_to_match_regex",
    "kwargs": {
      "column": "invoice_no",
      "regex": "^I\\d+$",
      "batch_id": "retail-clean-data-retail-transactions-clean"
    }
  }
}


### Expectation 7: Column Values To Not Match Regex (`customer_id`)

No value in `customer_id` should consist of only whitespace or empty strings. Such values would pass a standard null check but are effectively useless, which represent "hidden empty" values that look non-null but contain no actual data.

In [11]:
result_7 = validator.expect_column_values_to_not_match_regex(
    'customer_id',
    r'^\s*$'
)
print(result_7)

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  },
  "success": true,
  "meta": {},
  "result": {
    "element_count": 99457,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "expectation_config": {
    "meta": {},
    "expectation_type": "expect_column_values_to_not_match_regex",
    "kwargs": {
      "column": "customer_id",
      "regex": "^\\s*$",
      "batch_id": "retail-clean-data-retail-transactions-clean"
    }
  }
}


### Expectation 8: Column Values To Not Be Null (`price`)

No value in `price` should be null or missing. Since `price` is used to calculate total revenue (`quantity × price`), a null price would produce incorrect revenue figures and misleading dashboard insights.

In [12]:
result_8 = validator.expect_column_values_to_not_be_null('price')
print(result_8)

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

{
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  },
  "success": true,
  "meta": {},
  "result": {
    "element_count": 99457,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": []
  },
  "expectation_config": {
    "meta": {},
    "expectation_type": "expect_column_values_to_not_be_null",
    "kwargs": {
      "column": "price",
      "batch_id": "retail-clean-data-retail-transactions-clean"
    }
  }
}


### Expectation 9: Column Values To Be Dateutil Parseable (`invoice_date`)

All values in `invoice_date` must be parseable as valid dates. Invalid date strings (e.g., `32/13/2023` or `not-a-date`) would cause failures in Kibana time-series visualizations and Airflow date-based scheduling.

In [13]:
result_9 = validator.expect_column_values_to_be_dateutil_parseable('invoice_date')
print(result_9)

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  },
  "success": true,
  "meta": {},
  "result": {
    "element_count": 99457,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "expectation_config": {
    "meta": {},
    "expectation_type": "expect_column_values_to_be_dateutil_parseable",
    "kwargs": {
      "column": "invoice_date",
      "batch_id": "retail-clean-data-retail-transactions-clean"
    }
  }
}


## 6. Save Expectation Suite

In [14]:
validator.save_expectation_suite(discard_failed_expectations=False)
print("Expectation suite saved successfully!")

Expectation suite saved successfully!


## 7. Checkpoint

In [15]:
checkpoint = context.add_or_update_checkpoint(
    name='retail_validation_checkpoint',
    validator=validator,
)
print("Checkpoint created!")

Checkpoint created!


In [16]:
checkpoint_result = checkpoint.run()
print("Checkpoint run complete!")
print(f"Success: {checkpoint_result.success}")

Calculating Metrics:   0%|          | 0/49 [00:00<?, ?it/s]

Checkpoint run complete!
Success: True


## 8. Build Data Docs

In [17]:
context.build_data_docs()
print("Data Docs built! Open great_expectations/uncommitted/data_docs/local_site/index.html to view.")

Data Docs built! Open great_expectations/uncommitted/data_docs/local_site/index.html to view.
